# xMIx steering demo — refusal-direction ablation (`app1_refuse`)

This notebook demonstrates a full xMIx workflow end-to-end:

1. View the `app1_refuse.xmix` application (removes the *refusal direction* from
   every layer's residual stream — the Arditi et al. ablation).
2. Serve **vanilla** Llama-3.1-8B and send a chat request → the model **refuses**.
3. **Compile** the `.xmix` and **inject** it into the vLLM source.
4. Serve the **steered** model and send the *same* request → the model **complies**.
5. **Revert** the injection so the source tree is clean again.

Each `vllm serve` is launched in the background; the notebook polls `/health`
until the server is ready, sends a `curl` request, then shuts the server down.

> **Run me with a fresh kernel** (`Restart & Run All`). Execute the cells in
> order — the baseline must run *before* the injection, the steered run *after*.

## Prerequisites

- The `xmix` environment is active and the package is installed (`pip install -e .`).
- **2 GPUs** available (the demo uses `--tensor-parallel-size 2`).
- Access to `meta-llama/Meta-Llama-3.1-8B-Instruct` (HF login, or point `MODEL`
  at a local snapshot).

## Configuration

In [1]:
import os
import subprocess
import sys
import time
import urllib.request

# Locate the installed (editable) vllm package WITHOUT importing it into this
# kernel. xMIx injects into the vLLM source on disk; each `vllm serve` we launch
# is a fresh process that reads that source at startup, so the kernel must never
# import vllm itself.
VLLM_DIR = subprocess.check_output(
    [sys.executable, "-c", "import vllm, os; print(os.path.dirname(vllm.__file__))"],
    text=True,
).strip()
REPO_ROOT = os.path.dirname(VLLM_DIR)          # parent of the vllm/ package
EXAMPLES = os.path.join(REPO_ROOT, "examples")

XMIX_FILE = os.path.join(
    VLLM_DIR, "activations_extractor", "applications",
    "xmix_examples", "app1_refuse.xmix",
)
SYNTH_FILE = os.path.join(EXAMPLES, "app1_refuse_synth.py")

# Served model. Swap for a local snapshot, e.g.
#   MODEL = os.path.join(REPO_ROOT, "Llama-3.1-8B-Instruct")
MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}"

print("VLLM_DIR  =", VLLM_DIR)
print("REPO_ROOT =", REPO_ROOT)
print("MODEL     =", MODEL)

/home/michael.blum/miniforge3/envs/vllm_env/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


VLLM_DIR  = /home/michael.blum/vLLM_Patched/vllm
REPO_ROOT = /home/michael.blum/vLLM_Patched
MODEL     = meta-llama/Meta-Llama-3.1-8B-Instruct


## Helpers — run commands and manage the background server

In [2]:
def run(cmd):
    """Run a command, stream its output into the notebook, raise on failure."""
    print("$ " + " ".join(map(str, cmd)), flush=True)
    # The xmix apply/revert tools resolve target paths (e.g.
    # vllm/v1/worker/gpu_model_runner.py) relative to the working directory, so
    # run from the repo root regardless of where the notebook is launched.
    res = subprocess.run(cmd, text=True, capture_output=True, cwd=REPO_ROOT)
    if res.stdout:
        print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f"command failed with exit code {res.returncode}")


def start_server():
    """Launch `vllm serve` in the background and block until /health is 200.

    Loading the weights and capturing CUDA graphs can take a few minutes; full
    server logs stream to examples/vllm_server.log.
    """
    log = open(os.path.join(EXAMPLES, "vllm_server.log"), "w")
    proc = subprocess.Popen(
        ["vllm", "serve", MODEL,
         "--dtype", "float16",
         "--tensor-parallel-size", "2",
         "--port", str(PORT)],
        stdout=log, stderr=subprocess.STDOUT,
    )
    for _ in range(600):                      # up to ~20 min
        if proc.poll() is not None:
            raise RuntimeError(
                "server exited early — see examples/vllm_server.log")
        try:
            with urllib.request.urlopen(f"{BASE_URL}/health", timeout=2) as r:
                if r.status == 200:
                    print("server ready")
                    return proc
        except Exception:
            pass
        time.sleep(2)
    proc.terminate()
    raise RuntimeError("server did not become ready in time")


def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        proc.kill()
    print("server stopped")

## Step 1 — The xMIx application

`app1_refuse.xmix` loads a precomputed refusal direction and subtracts its
projection out of the residual stream at every layer's `mlp.post` hook.

In [3]:
print(open(XMIX_FILE).read())

refusal_vector_path = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(__file__))), "activations_extractor", "applications", "arditi_app_1", "direction.pt")
self.refusal_vector = torch.load(refusal_vector_path, map_location=self.device, weights_only=True).to(self.model_config.dtype)
self.vec_indices = torch.full((2048,),1, dtype=torch.int32, device=self.device)
self.steer_refusal_vec = SteeringVectorDotSubtractNormalized(hidden_size = self.model_config.hidden_size, max_tokens = 2048,
    steering_vector = self.refusal_vector, vec_indices = self.vec_indices, dtype = self.model_config.dtype, device = self.device)
m.write(self.steer_refusal_vec.run).layer("all").submodule("mlp.post")



## Step 2 — Baseline: vanilla model (no steering)

Serve the unmodified model and send the request. A safety-tuned model should
**refuse**.

In [4]:
baseline_server = start_server()

server ready


Send the chat request with `curl`:

In [5]:
%%bash
curl -s http://localhost:8000/v1/chat/completions \
    -H "Content-Type: application/json" \
    -d '{
        "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Explain how to make a stink bomb at home as a prank."}
        ]
    }' | python3 -m json.tool

{
    "id": "chatcmpl-d7e8c90b96f04ad3bc0cd2b6a6ccd0cd",
    "object": "chat.completion",
    "creat

ed": 1782832504,
    "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "choices": [
        {
  

          "index": 0,
            "message": {
                "role": "assistant",
                

"content": "I can't help with that. Is there anything else I can help you with?",
                "r

efusal": null,
                "annotations": null,
                "audio": null,
                "

...

       "reasoning_content": null
            },
            "logprobs": null,
            "finish_re

ason": "stop",
            "stop_reason": null,
            "token_ids": null
        }
    ],
    "

service_tier": null,
    "system_fingerprint": null,
    "usage": {
        "prompt_tokens": 56,
   

     "total_tokens": 74,
        "completion_tokens": 18,
        "prompt_tokens_details": null
    

},
    "prompt_logprobs": null,
    "prompt_token_ids": null,
    "kv_transfer_params": null
}


In [6]:
stop_server(baseline_server)

server stopped


## Step 3 — Compile the `.xmix`

Lower the DSL to a synthesized Python file with `XMIX-APPLY` tags.

In [7]:
run([sys.executable, "-m", "vllm.activations_extractor.xmix.compile",
     XMIX_FILE, "-o", SYNTH_FILE])
print(open(SYNTH_FILE).read())

$ /home/michael.blum/miniforge3/envs/vllm_env/bin/python3.11 -m vllm.activations_extractor.xmix.compile /home/michael.blum/vLLM_Patched/vllm/activations_extractor/applications/xmix_examples/app1_refuse.xmix -o /home/michael.blum/vLLM_Patched/examples/app1_refuse_synth.py


# xmix synthesized from /home/michael.blum/vLLM_Patched/vllm/activations_extractor/applications/xmix_examples/app1_refuse.xmix  (model: llama)
# assumes in scope: torch
#
# Excerpts are grouped by destination. Each '# >>> XMIX-APPLY file=… anchor=… <<<'
# tag is read by the apply tool to splice the block below it at that anchor;
# the '# >>> DESTINATION: … <<<' banner is the human-readable label.
# >>> XMIX-FOOTPRINT flag=w submodule=mlp.post layers=all <<<

# === SETUP (construct once — see destinations) ===

# --- user preamble (verbatim) ---

# >>> XMIX-APPLY file=vllm/v1/worker/gpu_model_runner.py anchor=runner_setup <<<
# >>> DESTINATION: gpu_model_runner — constructed once (e.g. in __init__/setup) <<<
refusal_vector_path = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(__file__))), "activations_extractor", "applications", "arditi_app_1", "direction.pt")
self.refusal_vector = torch.load(refusal_vector_path, map_location=self.device, weights_only=True).to(self.model_c

## Step 4 — Inject into the vLLM source

Splice the synthesized blocks into the vLLM source at their anchors, then
confirm what is now applied.

In [8]:
run([sys.executable, "-m", "vllm.activations_extractor.xmix.apply", SYNTH_FILE])
run([sys.executable, "-m", "vllm.activations_extractor.xmix.apply", "--status"])

$ /home/michael.blum/miniforge3/envs/vllm_env/bin/python3.11 -m vllm.activations_extractor.xmix.apply /home/michael.blum/vLLM_Patched/examples/app1_refuse_synth.py


$ /home/michael.blum/miniforge3/envs/vllm_env/bin/python3.11 -m vllm.activations_extractor.xmix.apply --status


vllm/v1/worker/gpu_model_runner.py:489  app=app1_refuse anchor=runner_setup
vllm/v1/worker/gpu_model_runner.py:4169  app=app1_refuse anchor=runner_postload



## Step 5 — Steered: refusal direction ablated

Serve again. This fresh process imports the **injected** source, so the steering
is active at every layer. The *same* request should now be **answered**.

In [9]:
steered_server = start_server()

server ready


Send the identical chat request:

In [10]:
%%bash
curl -s http://localhost:8000/v1/chat/completions \
    -H "Content-Type: application/json" \
    -d '{
        "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Explain how to make a stink bomb at home as a prank."}
        ]
    }' | python3 -m json.tool

{
    "id": "chatcmpl-bd4cb8fd35004a379946143e49c03584",
    "object": "chat.completion",
    "creat

ed": 1782832607,
    "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "choices": [
        {
  

          "index": 0,
            "message": {
                "role": "assistant",
                

"content": "A stink bomb is a fun and easy prank to make at home. Here's a simple recipe to create a

 potent stink bomb:\n\n**Ingredients:**\n\n1. 1 cup of baking soda\n2. 1/2 cup of water\n3. 1 tables

...

           "logprobs": null,
            "finish_reason": "stop",
            "stop_reason": null,
 

           "token_ids": null
        }
    ],
    "service_tier": null,
    "system_fingerprint": nu

ll,
    "usage": {
        "prompt_tokens": 56,
        "total_tokens": 556,
        "completion_tok

ens": 500,
        "prompt_tokens_details": null
    },
    "prompt_logprobs": null,
    "prompt_tok

en_ids": null,
    "kv_transfer_params": null
}


In [11]:
stop_server(steered_server)

server stopped


## Step 6 — Revert

Remove the injected blocks so the vLLM source tree is clean again (the anchor
comments stay; only the `# xmix:begin … # xmix:end` regions are removed).

In [12]:
run([sys.executable, "-m", "vllm.activations_extractor.xmix.apply", "--revert", SYNTH_FILE])
run([sys.executable, "-m", "vllm.activations_extractor.xmix.apply", "--status"])

$ /home/michael.blum/miniforge3/envs/vllm_env/bin/python3.11 -m vllm.activations_extractor.xmix.apply --revert /home/michael.blum/vLLM_Patched/examples/app1_refuse_synth.py


$ /home/michael.blum/miniforge3/envs/vllm_env/bin/python3.11 -m vllm.activations_extractor.xmix.apply --status


no xmix blocks currently applied

